In [13]:
### Necessary Imports
import pandas as pd
from FeatureSelection import BestFeature as bf
from ModelSelection import ModelComparisonUtility as mcu
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
import joblib
## Reading the data from CSV file
dataset=bf.importDataset() 

#### Function to Evaluate any Feature Set to

In [2]:

X, y = bf.preprocess_data(df=dataset,target_column="Loan_Status",drop_columns=["Loan_ID"])

results = bf.find_best_feature_selection(X, y, k=5)
print(results)

                                                             features  \
Forward Selection   [ApplicantIncome, CoapplicantIncome, Dependent...   
SelectKBest         [ApplicantIncome, CoapplicantIncome, Education...   
Backward Selection  [ApplicantIncome, CoapplicantIncome, Education...   
Feature Importance  [ApplicantIncome, LoanAmount, CoapplicantIncom...   
RFE                 [ApplicantIncome, CoapplicantIncome, Married_Y...   

                   cv_accuracy  
Forward Selection     0.760236  
SelectKBest            0.75942  
Backward Selection     0.75942  
Feature Importance    0.757699  
RFE                   0.753623  


## Stability table to Identify Recurring common Columns across features

In [3]:
## This table helps to identify which Featu
stability_table = bf.build_feature_stability_table(results)
print(stability_table)

                     Feature  Frequency  \
0            ApplicantIncome          5   
1          CoapplicantIncome          5   
2  Credit_History_No_History          5   
3     Education_Not Graduate          3   
4          Self_Employed_Yes          3   
5                Married_Yes          2   
6               Dependents_2          1   
7                 LoanAmount          1   

                                    Methods_Selected  
0  Forward Selection, SelectKBest, Backward Selec...  
1  Forward Selection, SelectKBest, Backward Selec...  
2  Forward Selection, SelectKBest, Backward Selec...  
3               SelectKBest, Backward Selection, RFE  
4  Forward Selection, SelectKBest, Backward Selec...  
5                            Feature Importance, RFE  
6                                  Forward Selection  
7                                 Feature Importance  


#### Using Select K algortithm for further Development 

##### Finding Best K

In [4]:
## Initalize K value & find the best K
k_values = range(2, min(15, X.shape[1]) + 1)
k_results = bf.tune_k_select_k_best(X, y,k_range=range(2, min(15, X.shape[1]) + 1))

print(k_results)

     k  mean_accuracy  std_accuracy
0    2       0.709692      0.000000
1    3       0.739312      0.008644
2    4       0.738768      0.009172
3    5       0.759420      0.012549
4    6       0.758152      0.013179
5    7       0.759511      0.011993
6    8       0.759873      0.012910
7    9       0.763315      0.010857
8   10       0.764130      0.012218
9   11       0.764493      0.013238
10  12       0.764946      0.013395
11  13       0.764764      0.014064
12  14       0.764583      0.014057
13  15       0.765942      0.014497


##### Choose the Best K

In [5]:
threshold = k_results["mean_accuracy"].max() - 0.002
best_k = k_results[k_results["mean_accuracy"] >= threshold]["k"].min()
best_k

10

##### Final Best K Columns

In [6]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=best_k)
selector.fit(X, y)

best_10_columns = X.columns[selector.get_support()].tolist()
best_10_columns

['ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Gender_Male',
 'Married_Yes',
 'Dependents_2',
 'Education_Not Graduate',
 'Self_Employed_Yes',
 'Property_Area_Semiurban',
 'Credit_History_No_History']

##### Note: We will use the above columns to train the model

In [18]:
# FINAL MODEL (trained ONLY on best_10_columns)
final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GradientBoostingClassifier(random_state=42))
])

final_pipeline.fit(X_best_10, y)

# SAVE MODEL + FEATURE NAMES TOGETHER
joblib.dump(
    {
        "model": final_pipeline,
        "features": best_10_columns
    },
    "loan_final_model_bundle.sav"
)

print("Model trained and saved using best_10_columns")


Model trained and saved using best_10_columns


In [16]:
joblib.dump(
    {
        "model": final_pipeline,
        "features": best_10_columns
    },
    "loan_final_model_bundle.sav"
)

print("Final model trained on best_10_columns and saved")

Final model trained on best_10_columns and saved
